# AquaSense — Water-quality forecasting + early alerts (Colab)

Runs the AquaSense prediction layer end to end on the Udanor IoTPond data:

1. **Data preparation**: clean the 20-s sensor logs into a 10-min grid with lag features
2. **Model comparison**: Persistence, Linear trend, Ridge, Gradient boosting
3. **Evaluation**: RMSE, MAE, MAPE, R², skill vs persistence, per horizon step
4. **Early-alert scores**: precision, recall, **F1**, breaches warned, lead time
5. **Train + download** the final models for the dashboard's forecasting service

Run the cells top to bottom (**Runtime → Run all**). A CPU runtime is enough. At the
default 10-min horizon the whole notebook takes about 2–3 min; with a 3-h horizon
(`HORIZON_STEPS = 18`) gradient boosting fits 18 models per parameter (~15 min).

## 0. Setup

In [ ]:
# Same scikit-learn version as the local project, so the downloaded models load there.
!pip install -q "scikit-learn==1.9.1" pyyaml joblib

import os, sys
ROOT = "/content/aquasense"
for d in ["prediction", "controller", "data"]:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("working in", os.getcwd())

## 0b. Get the data

**Option A (default):** run the cell and pick the `IoTPond*.csv` files, or one `.zip` of them.

**Option B:** keep them in Google Drive: set `DRIVE_PATH` to the folder (or `.zip`) and the upload is skipped.

In [ ]:
import glob, shutil, zipfile

DRIVE_PATH = None   # e.g. "/content/drive/MyDrive/AquaSense/data"  or  ".../IoTPond.zip"

def _collect(path):
    if path.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            for m in z.namelist():
                if m.lower().endswith(".csv") and "pond" in m.lower() and not m.startswith("__MACOSX"):
                    with z.open(m) as src, open(os.path.join("data", os.path.basename(m)), "wb") as dst:
                        shutil.copyfileobj(src, dst)
    elif os.path.isdir(path):
        for f in glob.glob(os.path.join(path, "*[Pp]ond*.csv")):
            shutil.copy(f, "data")
    elif path.lower().endswith(".csv"):
        shutil.move(path, os.path.join("data", os.path.basename(path)))

if DRIVE_PATH:
    from google.colab import drive
    drive.mount("/content/drive")
    _collect(DRIVE_PATH)
elif not glob.glob("data/*[Pp]ond*.csv"):
    from google.colab import files
    for name in files.upload():
        _collect(name)

print(sorted(os.listdir("data")))

## 1. Prediction code

The same modules as `water_quality/prediction/` in the project, plus the controller's
`config.yaml` (the safe limits come from there). Just run these cells.

In [ ]:
%%writefile prediction/__init__.py
"""AquaSense prediction layer: forecasts water-quality parameters and raises
early (lead-time) alerts before a value leaves its safe range.

Kept separate from the controller's threshold engine (controller/engines) so a
cause-analysis layer can later sit on top of both.

    settings.py   one place for interval, lag window and horizon
    data_prep.py  raw logs -> clean fixed-interval grid -> lag-feature dataset
"""

In [ ]:
%%writefile prediction/settings.py
"""Configuration for the prediction layer.

Interval, lag window and horizon are all configurable; everything downstream
derives its step counts from `PrepConfig`.
"""
from __future__ import annotations

from dataclasses import dataclass, field

# The four parameters the monitoring baseline logs and we forecast.
PARAMS = ["do", "ph", "ammonia", "temperature"]

PRETTY = {"do": "Dissolved oxygen", "ph": "pH", "ammonia": "Ammonia",
          "temperature": "Temperature"}
UNITS = {"do": "mg/L", "ph": "", "ammonia": "mg/L", "temperature": "°C"}

# Biologically plausible ranges for water that holds LIVE fish. Stricter than
# the sensor's electrical range: values outside are probe faults, not water.
#   temperature  DS18B20 reports -127 when unplugged
#   do           > 20 mg/L is far beyond saturation at 22-30 °C; 0.0 = dead probe
#   ph           IoTPond probes drift to negative / ~3 after August (fish alive)
#   ammonia      IoTPond logs up to 1e8; > 10 mg/L TAN for weeks is not survivable
PLAUSIBLE = {
    "temperature": (10.0, 40.0),
    "do": (0.05, 20.0),
    "ph": (4.5, 10.0),
    "ammonia": (0.0, 10.0),
}

# A sensor reporting the exact same value for this long is stuck, not stable.
# Temperature (0.0625 °C resolution) can legitimately sit still for a while.
STUCK_HOURS = {"temperature": 12.0, "do": 1.0, "ph": 1.0, "ammonia": 1.0}

# Raw readings a bin needs before its median is trusted. DO and ammonia swing
# by several mg/L between 20-s samples, so one reading says little; temperature
# and pH are smooth, and the logger often drops to ~1 reading per bin at night.
MIN_SAMPLES_PER_BIN = {"temperature": 1, "ph": 1, "do": 3, "ammonia": 3}


@dataclass
class PrepConfig:
    freq_min: int = 10            # resample interval (minutes)
    lag_steps: int = 12           # past readings used as features (12 x 10 min = 2 h)
    horizon_steps: int = 1        # steps forecast ahead (1 x 10 min = 10 min)
    gap_fill_min: int = 30        # interpolate gaps up to this long; longer stay NaN
    hampel_window: int = 7        # trailing window (steps) for spike removal
    hampel_sigmas: float = 3.0
    smooth_steps: int = 3         # trailing rolling-median denoise; 1 = off
    min_pond_rows: int = 5000     # skip ponds with less raw data than this
    params: list[str] = field(default_factory=lambda: list(PARAMS))

    @property
    def freq(self) -> str:
        return f"{self.freq_min}min"

    @property
    def gap_fill_steps(self) -> int:
        return self.gap_fill_min // self.freq_min

In [ ]:
%%writefile prediction/data_prep.py
"""Step 1 — Data preparation for forecasting.

    raw logs (IoTPond CSVs or the controller's SQLite `readings` table)
      -> clean_raw:  plausibility filter, stuck-sensor removal, duplicate ponds
      -> resample:   fixed-interval median bins, spike filter, short-gap fill,
                     optional denoise
      -> make_lag_dataset: one row per time t with the past `lag_steps` values
                     of every parameter (features) and the next `horizon_steps`
                     values of the target (labels)

Every filter on the grid is TRAILING (looks only at the past), so the exact
same cleaning can run live on the backend without peeking into the future.

Run:
    python -m prediction.data_prep                      # IoTPond CSVs in ./data
    python -m prediction.data_prep --db controller/controller.db --tank pond1
"""
from __future__ import annotations

import argparse
import glob
import os
import re
import sqlite3

import numpy as np
import pandas as pd

from .settings import MIN_SAMPLES_PER_BIN, PARAMS, PLAUSIBLE, PRETTY, STUCK_HOURS, PrepConfig

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)                       # water_quality/
DATA_DIR = os.path.join(ROOT, "data")
OUT_DIR = os.path.join(ROOT, "outputs", "prediction")

# Smallest MAD the spike filter trusts, roughly one sensor resolution step.
# Without a floor, a flat stretch (MAD = 0) would flag any change as a spike.
HAMPEL_MIN_MAD = {"temperature": 0.1, "do": 0.1, "ph": 0.02, "ammonia": 0.01}


# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------
def _canonical_name(col: str) -> str | None:
    """'Dissolved Oxygen(g/ml)', 'DISOLVED OXYGEN', 'PH', ... -> canonical name."""
    base = str(col).split("(")[0].strip().lower()
    letters = re.sub(r"[^a-z]", "", base)
    if letters == "createdat" or base == "unnamed: 0":   # pond 8: headerless col 0
        return "time"
    if "temp" in letters:
        return "temperature"
    if "oxygen" in letters or "disolved" in letters or "dissolved" in letters:
        return "do"
    if letters == "ph":
        return "ph"
    if "ammonia" in letters:
        return "ammonia"
    return None


_TS = re.compile(r"(\d{4}-\d{2}-\d{2})[ T](\d{2}:\d{2}:\d{2})")


def _series_key(name: str) -> int:
    """Numeric id in a series name, so pond2 sorts before pond10."""
    m = re.search(r"(\d+)", str(name))
    return int(m.group(1)) if m else 0


def load_csv_dir(data_dir: str = DATA_DIR) -> pd.DataFrame:
    """Read every IoTPond*.csv into one long frame: series, time, <params>.

    Timezone suffixes (CET/UTC/+01:00) are dropped; each pond is internally
    consistent, which is all lag features need. Date-only rows ('5/12/2021')
    cannot be placed on a grid and are dropped.
    """
    files = sorted(glob.glob(os.path.join(data_dir, "*[Pp]ond*.csv")),
                   key=lambda p: _series_key(os.path.basename(p)))
    if not files:
        raise FileNotFoundError(f"No IoTPond*.csv files in {data_dir}")
    frames = []
    for path in files:
        raw = pd.read_csv(path, dtype=str, low_memory=False)
        mapping = {}
        for col in raw.columns:
            name = _canonical_name(col)
            if name and name not in mapping.values():
                mapping[col] = name
        if "time" not in mapping.values():
            mapping[raw.columns[0]] = "time"
        df = raw[list(mapping)].rename(columns=mapping)
        parts = df["time"].str.extract(_TS)
        df["time"] = pd.to_datetime(parts[0] + " " + parts[1],
                                    format="%Y-%m-%d %H:%M:%S", errors="coerce")
        for p in PARAMS:
            df[p] = (pd.to_numeric(df[p], errors="coerce").replace([np.inf, -np.inf], np.nan)
                     if p in df else np.nan)
        df["series"] = f"pond{_series_key(os.path.basename(path))}"
        frames.append(df.dropna(subset=["time"])[["series", "time"] + PARAMS])
    return pd.concat(frames, ignore_index=True)


def load_controller_db(db_path: str, tank: str | None = None,
                       sources: tuple[str, ...] = ("payload",),
                       since: float | None = None) -> pd.DataFrame:
    """Read the controller's long-format `readings` table into the same shape.

    Only real sensor values are used by default (`source='payload'`): training
    on the server's simulated values would teach the model the simulator.
    FAULT readings are skipped; clean_raw applies its own plausibility filter.
    `since` (epoch seconds) limits the read to recent rows for live forecasts.
    """
    q = (f"SELECT tank_id, ts, parameter, value FROM readings "
         f"WHERE status != 'FAULT' AND value IS NOT NULL "
         f"AND parameter IN ({','.join('?' * len(PARAMS))}) "
         f"AND source IN ({','.join('?' * len(sources))})")
    args: list = [*PARAMS, *sources]
    if tank:
        q += " AND tank_id = ?"
        args.append(tank)
    if since is not None:
        q += " AND ts >= ?"
        args.append(since)
    # read-only: the controller owns this database
    with sqlite3.connect(f"file:{db_path}?mode=ro", uri=True) as conn:
        long = pd.read_sql_query(q, conn, params=args)
    if long.empty:
        return pd.DataFrame(columns=["series", "time"] + PARAMS)
    wide = long.pivot_table(index=["tank_id", "ts"], columns="parameter",
                            values="value", aggfunc="mean").reset_index()
    wide = wide.rename(columns={"tank_id": "series"})
    wide["time"] = pd.to_datetime(wide["ts"], unit="s")
    for p in PARAMS:
        if p not in wide:
            wide[p] = np.nan
    return wide[["series", "time"] + PARAMS]


# ---------------------------------------------------------------------------
# Clean raw readings
# ---------------------------------------------------------------------------
def _stuck_mask(times: pd.Series, values: pd.Series, max_hours: float) -> pd.Series:
    """True where a value belongs to a run of identical readings lasting longer
    than `max_hours` (a frozen sensor, not stable water)."""
    run = (values != values.shift()).cumsum()
    by_run = times.groupby(run)
    span = by_run.transform("max") - by_run.transform("min")
    return values.notna() & (span > pd.Timedelta(hours=max_hours))


def clean_raw(raw: pd.DataFrame, cfg: PrepConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Drop thin series and duplicated rows, then null out implausible and
    stuck readings. Returns (clean, per-series report of what was removed)."""
    df = raw.sort_values(["series", "time"], key=lambda s: s.map(_series_key)
                         if s.name == "series" else s).reset_index(drop=True)

    counts = df.groupby("series").size()
    thin = counts[counts < cfg.min_pond_rows].index
    df = df[~df["series"].isin(thin)]

    # A row identical (time + every value) to one already seen in an earlier
    # series is a copy: IoTPond2 repeats IoTpond1 from July onward.
    dup = df.duplicated(subset=["time"] + PARAMS, keep="first") & df[PARAMS].notna().any(axis=1)
    report = pd.DataFrame({"raw_rows": df.groupby("series").size(),
                           "duplicate_rows": dup.groupby(df["series"]).sum()})
    df = df[~dup].reset_index(drop=True)

    for p in cfg.params:
        lo, hi = PLAUSIBLE[p]
        bad = (df[p] < lo) | (df[p] > hi)
        report[f"{p}_implausible_%"] = 100 * bad.groupby(df["series"]).mean()
        df.loc[bad, p] = np.nan

        stuck = pd.Series(False, index=df.index)
        for _, g in df.groupby("series"):
            stuck.loc[g.index] = _stuck_mask(g["time"], g[p], STUCK_HOURS[p])
        report[f"{p}_stuck_%"] = 100 * stuck.groupby(df["series"]).mean()
        df.loc[stuck, p] = np.nan

    for s in thin:
        report.loc[s, "raw_rows"] = counts[s]
    report["kept"] = ~report.index.isin(thin)
    return df, report.round(1)


# ---------------------------------------------------------------------------
# Resample onto a fixed grid
# ---------------------------------------------------------------------------
def _hampel_trailing(x: pd.Series, window: int, sigmas: float, min_mad: float) -> pd.Series:
    """Null out spikes: points further than `sigmas` robust std-devs from the
    median of the trailing window."""
    med = x.rolling(window, min_periods=3).median()
    mad = (x - med).abs().rolling(window, min_periods=3).median().clip(lower=min_mad)
    spike = (x - med).abs() > sigmas * 1.4826 * mad
    return x.mask(spike)


def _fill_short_gaps(x: pd.Series, max_steps: int) -> pd.Series:
    """Linearly interpolate NaN runs of at most `max_steps`; longer outages are
    left untouched (pandas' `limit` would half-fill them instead)."""
    if max_steps <= 0:
        return x
    isna = x.isna()
    run = (isna != isna.shift()).cumsum()
    run_len = isna.groupby(run).transform("sum")
    filled = x.interpolate(method="linear", limit_area="inside")
    return x.where(~(isna & (run_len <= max_steps)), filled)


def resample(clean: pd.DataFrame, cfg: PrepConfig) -> pd.DataFrame:
    """Per series: median of raw readings per `freq` bin (robust to the 20-s
    sensor noise), drop bins with too few raw samples, remove spikes, fill
    short gaps, then denoise with a trailing rolling median.

    Output: series, time, <params>, n_raw, <param>_filled
    """
    out = []
    for series, g in clean.groupby("series", sort=False):
        g = g.set_index("time")[cfg.params]
        binned = g.resample(cfg.freq)
        grid = binned.median()
        n = binned.count()
        for p in cfg.params:
            grid[p] = grid[p].mask(n[p] < MIN_SAMPLES_PER_BIN[p])

        for p in cfg.params:
            x = _hampel_trailing(grid[p], cfg.hampel_window, cfg.hampel_sigmas,
                                 HAMPEL_MIN_MAD[p])
            observed = x.notna()
            x = _fill_short_gaps(x, cfg.gap_fill_steps)
            if cfg.smooth_steps > 1:
                present = x.notna()
                x = x.rolling(cfg.smooth_steps, min_periods=1).median().where(present)
            grid[p] = x
            grid[f"{p}_filled"] = x.notna() & ~observed

        grid["n_raw"] = n.max(axis=1)
        grid["series"] = series
        out.append(grid.reset_index())
    cols = ["series", "time"] + cfg.params + ["n_raw"] + [f"{p}_filled" for p in cfg.params]
    return pd.concat(out, ignore_index=True)[cols]


# ---------------------------------------------------------------------------
# Supervised lag dataset
# ---------------------------------------------------------------------------
def make_lag_dataset(grid: pd.DataFrame, target: str, cfg: PrepConfig,
                     require_labels: bool = True) -> pd.DataFrame:
    """One row per (series, t).

    Features: <p>_lag0 .. <p>_lag{L-1} for every parameter (lag0 = value at t),
              plus time-of-day as sin/cos (DO and temperature follow a daily cycle).
    Labels:   y_h1 .. y_hH = target at t+1 .. t+H steps.

    A row needs its full target lag window (and, for training, every label).
    Other parameters' lags may be NaN: gradient boosting handles that, and the
    linear models only use the target's own lags.
    Shifts happen inside each series on a gap-preserving grid, so a window
    never spans two ponds or bridges a long outage.
    """
    frames = []
    for series, g in grid.groupby("series", sort=False):
        g = g.sort_values("time")
        cols = {}
        for p in cfg.params:
            for k in range(cfg.lag_steps):
                cols[f"{p}_lag{k}"] = g[p].shift(k)
        for h in range(1, cfg.horizon_steps + 1):
            cols[f"y_h{h}"] = g[target].shift(-h)
        sup = pd.DataFrame(cols, index=g.index)
        hour = g["time"].dt.hour + g["time"].dt.minute / 60
        sup["tod_sin"] = np.sin(2 * np.pi * hour / 24)
        sup["tod_cos"] = np.cos(2 * np.pi * hour / 24)
        sup.insert(0, "time", g["time"])
        sup.insert(0, "series", series)
        frames.append(sup)
    sup = pd.concat(frames, ignore_index=True)

    required = [f"{target}_lag{k}" for k in range(cfg.lag_steps)]
    if require_labels:
        required += [f"y_h{h}" for h in range(1, cfg.horizon_steps + 1)]
    return sup.dropna(subset=required).sort_values(["time", "series"]).reset_index(drop=True)


def label_matrix(sup: pd.DataFrame, cfg: PrepConfig) -> np.ndarray:
    """(n, H) array of the true future values y_h1 .. y_hH."""
    return sup[[f"y_h{h}" for h in range(1, cfg.horizon_steps + 1)]].to_numpy()


# ---------------------------------------------------------------------------
# Reporting
# ---------------------------------------------------------------------------
def grid_report(grid: pd.DataFrame, cfg: PrepConfig) -> pd.DataFrame:
    rows = {}
    for series, g in grid.groupby("series", sort=False):
        r = {"start": g["time"].min().strftime("%Y-%m-%d"),
             "end": g["time"].max().strftime("%Y-%m-%d"),
             "grid_rows": len(g)}
        for p in cfg.params:
            r[f"{p}_cov_%"] = 100 * g[p].notna().mean()
        rows[series] = r
    return pd.DataFrame(rows).T


def plot_before_after(raw: pd.DataFrame, grid: pd.DataFrame, cfg: PrepConfig,
                      series: str, days: int = 3, out_dir: str = OUT_DIR) -> str:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    g = grid[grid["series"] == series].set_index("time")
    start = g[cfg.params].dropna().index.min()
    end = start + pd.Timedelta(days=days)
    r = raw[(raw["series"] == series) & raw["time"].between(start, end)].set_index("time")
    g = g.loc[start:end]

    fig, axes = plt.subplots(len(cfg.params), 1, figsize=(12, 2.4 * len(cfg.params)), sharex=True)
    for ax, p in zip(axes, cfg.params):
        lo, hi = PLAUSIBLE[p]
        ax.plot(r.index, r[p].where(r[p].between(lo, hi)), color="#bbbbbb", lw=0.5,
                label="raw (20 s, plausible only)")
        ax.plot(g.index, g[p], color="#1f77b4", lw=1.3, label=f"clean ({cfg.freq})")
        ax.set_ylabel(PRETTY[p])
        ax.grid(alpha=0.3)
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].set_title(f"{series}: raw vs cleaned, first {days} days")
    fig.tight_layout()
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"prep_{series}.png")
    fig.savefig(path, dpi=120)
    plt.close(fig)
    return path


def prepare(source_df: pd.DataFrame, cfg: PrepConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    """raw -> clean grid. Returns (grid, cleaning report)."""
    clean, report = clean_raw(source_df, cfg)
    return resample(clean, cfg), report


def main() -> None:
    ap = argparse.ArgumentParser(description="AquaSense forecasting: data preparation")
    ap.add_argument("--data", default=DATA_DIR, help="folder with IoTPond*.csv")
    ap.add_argument("--db", help="controller SQLite DB instead of CSVs")
    ap.add_argument("--tank", help="tank id when reading --db")
    ap.add_argument("--freq", type=int, default=PrepConfig.freq_min, help="interval, minutes")
    ap.add_argument("--lags", type=int, default=PrepConfig.lag_steps)
    ap.add_argument("--horizon", type=int, default=PrepConfig.horizon_steps)
    ap.add_argument("--smooth", type=int, default=PrepConfig.smooth_steps)
    ap.add_argument("--out", default=OUT_DIR)
    args = ap.parse_args()
    cfg = PrepConfig(freq_min=args.freq, lag_steps=args.lags,
                     horizon_steps=args.horizon, smooth_steps=args.smooth)

    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 40)

    raw = load_controller_db(args.db, args.tank) if args.db else load_csv_dir(args.data)
    print(f"[load] {len(raw):,} raw rows from {raw['series'].nunique()} series")

    grid, report = prepare(raw, cfg)
    print("\n[clean] what was removed per series (% of that series' raw rows)")
    print(report.to_string())

    print(f"\n[grid] {cfg.freq} grid, gaps <= {cfg.gap_fill_min} min filled, "
          f"trailing median denoise = {cfg.smooth_steps} steps")
    print(grid_report(grid, cfg).round(1).to_string())

    print(f"\n[lags] usable supervised rows per target "
          f"(window {cfg.lag_steps} x {cfg.freq} = {cfg.lag_steps * cfg.freq_min / 60:g} h, "
          f"horizon {cfg.horizon_steps} x {cfg.freq} = {cfg.horizon_steps * cfg.freq_min / 60:g} h)")
    for p in cfg.params:
        sup = make_lag_dataset(grid, p, cfg)
        days = len(sup) * cfg.freq_min / 1440
        per = sup.groupby("series").size().to_dict()
        print(f"  {PRETTY[p]:<17} {len(sup):>7,} rows (~{days:,.0f} pond-days)  {per}")

    print("\n[sample] cleaned grid, pond with most data, first complete rows")
    best = grid.dropna(subset=cfg.params).groupby("series").size().idxmax()
    print(grid[grid["series"] == best].dropna(subset=cfg.params).head(8)
          .round({p: 3 for p in cfg.params}).to_string(index=False))
    print("\n[describe] cleaned values (all series)")
    print(grid[cfg.params].describe().round(3).to_string())

    os.makedirs(args.out, exist_ok=True)
    grid_path = os.path.join(args.out, "clean_grid.csv.gz")
    grid.to_csv(grid_path, index=False)
    report.to_csv(os.path.join(args.out, "cleaning_report.csv"))
    plot = plot_before_after(raw, grid, cfg, best, out_dir=args.out)
    print(f"\nwrote {grid_path}\nwrote {plot}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile prediction/models.py
"""Step 2 — Forecasting models, simplest first.

Every model forecasts ALL horizon steps at once (t+1 .. t+H) for one target
parameter, from a lag dataset built by data_prep.make_lag_dataset:

    fit(sup, target, cfg)  ->  self
    predict(sup)           ->  array (n_rows, H) of forecast values

    Persistence     y(t+h) = y(t). No training. The bar to beat.
    LinearTrend     straight line through the last `window` readings, extended.
    Ridge           linear regression on the target's own lag window.
    GBoost          gradient-boosted trees on all four parameters' lag windows
                    plus time of day; one model per horizon step.

The trained models predict the CHANGE from the current value, then add it back
(y(t+h) = y(t) + delta). Pond levels differ a lot (one pond's DO sits at 3,
another's at 10), but how values move over two hours is much more alike, so
learning the change transfers across tanks.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from .settings import PLAUSIBLE, PrepConfig


def _relative(sup: pd.DataFrame, params: list[str], lags: int) -> pd.DataFrame:
    """lag0 kept as the level; lag1..lagN re-expressed as (lag_k - lag0)."""
    out = {}
    for p in params:
        now = sup[f"{p}_lag0"]
        out[f"{p}_now"] = now
        for k in range(1, lags):
            out[f"{p}_d{k}"] = sup[f"{p}_lag{k}"] - now
    out["tod_sin"], out["tod_cos"] = sup["tod_sin"], sup["tod_cos"]
    return pd.DataFrame(out, index=sup.index)


class Forecaster:
    name = "base"

    def __init__(self, cfg: PrepConfig):
        self.cfg = cfg
        self.target: str | None = None

    def fit(self, sup: pd.DataFrame, target: str) -> "Forecaster":
        self.target = target
        return self

    def predict(self, sup: pd.DataFrame) -> np.ndarray:
        raise NotImplementedError

    def _labels(self, sup: pd.DataFrame) -> np.ndarray:
        return sup[[f"y_h{h}" for h in range(1, self.cfg.horizon_steps + 1)]].to_numpy()

    def _clip(self, pred: np.ndarray) -> np.ndarray:
        lo, hi = PLAUSIBLE[self.target]
        return np.clip(pred, lo, hi)


class Persistence(Forecaster):
    name = "Persistence"

    def predict(self, sup):
        now = sup[f"{self.target}_lag0"].to_numpy()
        return np.repeat(now[:, None], self.cfg.horizon_steps, axis=1)


class LinearTrend(Forecaster):
    """Least-squares slope over the last `window` steps, extrapolated."""
    name = "LinearTrend"

    def __init__(self, cfg, window: int = 6):
        super().__init__(cfg)
        self.window = min(window, cfg.lag_steps)

    def predict(self, sup):
        # x = 0, -1, ..., -(w-1) for lag0 .. lag{w-1}
        x = -np.arange(self.window, dtype=float)
        y = sup[[f"{self.target}_lag{k}" for k in range(self.window)]].to_numpy()
        xc = x - x.mean()
        slope = ((y - y.mean(axis=1, keepdims=True)) * xc).sum(axis=1) / (xc ** 2).sum()
        level = y.mean(axis=1) - slope * x.mean()          # fitted value at t
        h = np.arange(1, self.cfg.horizon_steps + 1)
        return self._clip(level[:, None] + slope[:, None] * h[None, :])


class RidgeLags(Forecaster):
    """Linear regression on the target's own lag window (+ time of day)."""
    name = "Ridge"

    def fit(self, sup, target):
        super().fit(sup, target)
        X = _relative(sup, [target], self.cfg.lag_steps)
        dy = self._labels(sup) - sup[f"{target}_lag0"].to_numpy()[:, None]
        self.model = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X, dy)
        return self

    def predict(self, sup):
        X = _relative(sup, [self.target], self.cfg.lag_steps)
        now = sup[f"{self.target}_lag0"].to_numpy()[:, None]
        # sklearn returns shape (n,) for a single-step horizon; keep (n, H)
        return self._clip(now + self.model.predict(X).reshape(len(X), -1))


class GBoost(Forecaster):
    """Gradient-boosted trees, one per horizon step, on every parameter's lags.
    Missing lags of the OTHER parameters are allowed (trees handle NaN).

    The other parameters enter only as recent CHANGES, not absolute levels:
    trees cannot extrapolate, and levels shift between tanks and when probes
    drift (IoTPond pH test levels sit below anything seen in training), which
    made the level-aware version clearly worse on pH and DO."""
    name = "GBoost"

    def __init__(self, cfg, max_iter: int = 300, seed: int = 42):
        super().__init__(cfg)
        self.max_iter, self.seed = max_iter, seed

    def _X(self, sup):
        X = _relative(sup, self.cfg.params, self.cfg.lag_steps)
        return X.drop(columns=[f"{p}_now" for p in self.cfg.params if p != self.target])

    def fit(self, sup, target):
        super().fit(sup, target)
        X = self._X(sup)
        dy = self._labels(sup) - sup[f"{target}_lag0"].to_numpy()[:, None]
        self.models = []
        for h in range(self.cfg.horizon_steps):
            m = HistGradientBoostingRegressor(
                max_iter=self.max_iter, learning_rate=0.05, max_leaf_nodes=31,
                min_samples_leaf=40, l2_regularization=1.0,
                early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
                random_state=self.seed)
            self.models.append(m.fit(X, dy[:, h]))
        return self

    def predict(self, sup):
        X = self._X(sup)
        now = sup[f"{self.target}_lag0"].to_numpy()
        return self._clip(np.column_stack([now + m.predict(X) for m in self.models]))


MODELS = {"persistence": Persistence, "trend": LinearTrend, "ridge": RidgeLags,
          "gboost": GBoost}

In [ ]:
%%writefile prediction/alerts.py
"""Step 3 — Lead-time (early) alerts from a forecast.

The baseline rule engine alerts when a value IS out of range. This module
alerts when the FORECAST leaves the range while the current value is still
safe, and says how long until it does:

    "Ammonia unsafe in ~2 h (forecast 0.62 mg/L > 0.5)"

A breach must hold for `confirm_steps` consecutive forecast steps, so one
noisy step does not raise an alert.

Alert margin: the forecast is compared with the limit moved inward by
`margin` (in the parameter's units). margin > 0 alerts when the forecast gets
CLOSE to the limit (catches more breaches); margin < 0 needs the forecast to be
clearly PAST it (fewer false alarms). `tune_margin` picks the value with the
best F1 on validation data; it never touches the safe limits themselves.

Safe limits are read from the controller's config.yaml: one source of truth
for the reactive engine and the predictive layer.
"""
from __future__ import annotations

import os
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd

from .settings import PRETTY, UNITS, PrepConfig

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
CONTROLLER_CONFIG = os.path.join(ROOT, "controller", "config.yaml")

CONFIRM_STEPS = 2


def load_safe_ranges(path: str = CONTROLLER_CONFIG) -> dict[str, tuple[float | None, float | None]]:
    import yaml

    with open(path) as f:
        params = yaml.safe_load(f)["parameters"]
    return {name: (p.get("min"), p.get("max")) for name, p in params.items()}


def shift_limits(limits: tuple, margin: float) -> tuple:
    """Move both limits inward by `margin` (outward when negative)."""
    lo, hi = limits
    return (None if lo is None else lo + margin, None if hi is None else hi - margin)


def _outside(values: np.ndarray, lo, hi) -> np.ndarray:
    out = np.zeros(values.shape, dtype=bool)
    if lo is not None:
        out |= values < lo
    if hi is not None:
        out |= values > hi
    return out


def first_breach_step(paths: np.ndarray, lo, hi, confirm: int = CONFIRM_STEPS) -> np.ndarray:
    """For each row of a (n, H) array of future values: the first step (1-based)
    that starts a run of `confirm` out-of-range steps, or 0 if none. A run that
    starts in the last steps of the horizon counts if it lasts to the end."""
    out = _outside(paths, lo, hi)
    n, H = out.shape
    run = np.zeros_like(out)
    for c in range(confirm):                 # run[:, h] = out[h] & ... & out[h+c]
        shifted = np.ones_like(out)
        shifted[:, :H - c] = out[:, c:]
        run = out & shifted if c == 0 else run & shifted
    hit = run.argmax(axis=1) + 1
    return np.where(run.any(axis=1), hit, 0)


@dataclass
class EarlyAlert:
    parameter: str
    side: str                 # low | high
    limit: float
    eta_min: int              # minutes until the forecast crosses the limit
    breach_at: str            # ISO time
    forecast_value: float     # forecast value at the breach step
    current_value: float
    message: str

    def to_dict(self) -> dict:
        return asdict(self)


def _eta_text(minutes: int) -> str:
    if minutes < 60:
        return f"~{minutes} min"
    hours = minutes / 60
    return f"~{hours:.0f} h" if abs(hours - round(hours)) < 0.25 else f"~{hours:.1f} h"


def early_alert(parameter: str, now: pd.Timestamp, current: float, forecast: np.ndarray,
                limits: tuple, cfg: PrepConfig, margin: float = 0.0) -> EarlyAlert | None:
    """Alert if `forecast` (H future values) breaches while `current` is safe."""
    lo, hi = limits
    if current is None or np.isnan(current) or _outside(np.array([current]), lo, hi)[0]:
        return None               # already out of range: the rule engine handles it
    a_lo, a_hi = shift_limits(limits, margin)
    step = int(first_breach_step(forecast[None, :], a_lo, a_hi)[0])
    if step == 0:
        return None
    value = float(forecast[step - 1])
    side, limit = ("low", lo) if a_lo is not None and value < a_lo else ("high", hi)
    eta = step * cfg.freq_min
    unit = UNITS[parameter]
    msg = (f"{PRETTY[parameter]} unsafe in {_eta_text(eta)} "
           f"(forecast {value:.2f}{' ' + unit if unit else ''}, "
           f"safe {'min' if side == 'low' else 'max'} {limit:g})")
    return EarlyAlert(parameter, side, float(limit), eta,
                      (now + pd.Timedelta(minutes=eta)).isoformat(), round(value, 3),
                      round(float(current), 3), msg)


# ---------------------------------------------------------------------------
# Evaluation of the alerts themselves (used by evaluate.py)
# ---------------------------------------------------------------------------
def alert_metrics(test: pd.DataFrame, y: np.ndarray, pred: np.ndarray, target: str,
                  limits: tuple, cfg: PrepConfig, margin: float = 0.0) -> dict:
    """How useful are the early alerts on held-out data?

    Only moments when the water is currently SAFE count (otherwise there is
    nothing to warn about early).

    precision   of the early alerts raised, the share followed by a real breach
                within the horizon
    recall      of the moments followed by a real breach within the horizon,
                the share that got an alert
    f1          2·P·R / (P + R): one score, high only if both are high
    events      distinct real breaches (series + breach time) in the test set
    warned_%    share of events that got at least one correct early warning
    lead_med_h  median, over warned events, of the EARLIEST correct warning
    warned_1h_% share of events warned at least 1 h ahead
    eta_mae_min how far off the predicted time-to-breach was, minutes
    """
    lo, hi = limits
    safe = ~_outside(test[f"{target}_lag0"].to_numpy(), lo, hi)
    actual = first_breach_step(y, lo, hi)[safe]
    alarm = first_breach_step(pred, *shift_limits(limits, margin))[safe]
    times = test["time"].to_numpy()[safe]
    series = test["series"].to_numpy()[safe]

    tp = (actual > 0) & (alarm > 0)
    step = pd.Timedelta(minutes=cfg.freq_min)
    ev = pd.DataFrame({"series": series[actual > 0],
                       "breach": times[actual > 0] + actual[actual > 0] * step,
                       "lead_steps": np.where(tp, actual, 0)[actual > 0]})
    per_event = ev.groupby(["series", "breach"])["lead_steps"].max()
    warned = per_event[per_event > 0]
    precision = tp.sum() / max((alarm > 0).sum(), 1)
    recall = tp.sum() / max((actual > 0).sum(), 1)
    return {
        "safe_rows": int(safe.sum()),
        "alerts": int((alarm > 0).sum()),
        "precision": precision,
        "recall": recall,
        "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "events": len(per_event),
        "warned_%": 100 * len(warned) / max(len(per_event), 1),
        "lead_med_h": float(warned.median() * cfg.freq_min / 60) if len(warned) else 0.0,
        "warned_1h_%": 100 * float((per_event * cfg.freq_min >= 60).mean()) if len(per_event) else 0.0,
        "eta_mae_min": float(np.abs(alarm[tp] - actual[tp]).mean() * cfg.freq_min) if tp.any() else np.nan,
    }


def tune_margin(val: pd.DataFrame, y: np.ndarray, pred: np.ndarray, target: str,
                limits: tuple, cfg: PrepConfig, n: int = 61) -> tuple[float, float]:
    """Alert margin with the best F1 on VALIDATION data (never the test set).

    Candidates span +/- the 95th percentile of the forecast error, so the search
    scales with each parameter's units. Ties go to the margin closest to 0;
    with no breaches in `val` (pH) the margin stays 0.
    Returns (margin, validation F1).
    """
    span = float(np.nanquantile(np.abs(pred - y), 0.95)) if len(y) else 0.0
    best, best_f1 = 0.0, alert_metrics(val, y, pred, target, limits, cfg)["f1"]
    for m in np.linspace(-span, span, n):
        f1 = alert_metrics(val, y, pred, target, limits, cfg, margin=m)["f1"]
        if f1 > best_f1 + 1e-9 or (abs(f1 - best_f1) <= 1e-9 and abs(m) < abs(best)):
            best, best_f1 = float(m), f1
    return best, best_f1

In [ ]:
%%writefile prediction/evaluate.py
"""Step 4 — Evaluate the step-2 models on held-out data.

Split: ONE chronological cutoff for all ponds. Test = the latest TEST_FRAC of
the timeline; train = rows whose last label (t + horizon) is still before the
cutoff, so no training label peeks into the test period. Nothing is shuffled.
One global cutoff (not one per pond) matters: ponds share a site and weather,
so training on pond A's September while testing pond B's September would leak.

Metrics per parameter x model x horizon step: RMSE, MAE, MAPE, R², and skill
(1 - RMSE / RMSE_persistence: > 0 means better than "nothing changes").

Early alerts: each model's alert margin (see alerts.py) is tuned for F1 on a
validation slice (latest 20% of train), then scored once on test. f1_untuned
is the same model alerting at the plain limit, for comparison.

Useful lead time: the longest horizon at which the best model still has
R² >= --min-r2 AND beats persistence (skill > 0).

Run:
    python -m prediction.evaluate
    python -m prediction.evaluate --freq 5 --lags 24 --horizon 36 --models persistence,ridge
"""
from __future__ import annotations

import argparse
import os
import time

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from .alerts import CONFIRM_STEPS, alert_metrics, load_safe_ranges, tune_margin
from .data_prep import (DATA_DIR, OUT_DIR, label_matrix, load_controller_db, load_csv_dir,
                        make_lag_dataset, prepare)
from .models import MODELS
from .settings import PRETTY, PrepConfig

TEST_FRAC = 0.2

# MAPE divides by the true value, so it explodes near zero (ammonia often reads
# 0.001). It is computed only where |true| >= this floor.
MAPE_FLOOR = {"do": 0.5, "ph": 1.0, "ammonia": 0.05, "temperature": 1.0}


def time_split(sup: pd.DataFrame, cfg: PrepConfig, test_frac: float = TEST_FRAC):
    cutoff = sup["time"].quantile(1 - test_frac)
    horizon = pd.Timedelta(minutes=cfg.horizon_steps * cfg.freq_min)
    return sup[sup["time"] + horizon < cutoff], sup[sup["time"] >= cutoff], cutoff


def horizon_metrics(y: np.ndarray, pred: np.ndarray, target: str) -> pd.DataFrame:
    rows = []
    for h in range(y.shape[1]):
        yt, yp = y[:, h], pred[:, h]
        m = np.abs(yt) >= MAPE_FLOOR[target]
        rows.append({"h": h + 1,
                     "rmse": float(np.sqrt(mean_squared_error(yt, yp))),
                     "mae": mean_absolute_error(yt, yp),
                     "mape_%": 100 * float(np.mean(np.abs((yt[m] - yp[m]) / yt[m]))) if m.any() else np.nan,
                     "r2": r2_score(yt, yp)})
    return pd.DataFrame(rows)


def evaluate_target(grid, target, cfg, model_names, limits):
    sup = make_lag_dataset(grid, target, cfg)
    train, test, cutoff = time_split(sup, cfg)
    # alert margins are chosen on the latest 20% of TRAIN, never on test
    inner, val, _ = time_split(train, cfg)
    y, y_val = label_matrix(test, cfg), label_matrix(val, cfg)
    frames, preds, alerts = [], {}, []
    for name in model_names:
        t0 = time.time()
        model = MODELS[name](cfg).fit(train, target)
        preds[name] = model.predict(test)
        m = horizon_metrics(y, preds[name], target)
        m.insert(0, "model", model.name)
        m["fit_s"] = round(time.time() - t0, 1)
        frames.append(m)
        tuner = MODELS[name](cfg).fit(inner, target)
        margin, _ = tune_margin(val, y_val, tuner.predict(val), target, limits, cfg)
        untuned = alert_metrics(test, y, preds[name], target, limits, cfg)["f1"]
        alerts.append({"target": target, "model": model.name, "margin": margin,
                       **alert_metrics(test, y, preds[name], target, limits, cfg, margin=margin),
                       "f1_untuned": untuned})
    # Oracle: alerts from the TRUE future. Its scores are the ceiling the data
    # allows (e.g. a breach after a sensor outage cannot be warned about).
    alerts.append({"target": target, "model": "(perfect forecast)", "margin": 0.0,
                   **alert_metrics(test, y, y, target, limits, cfg)})
    res = pd.concat(frames, ignore_index=True)
    base = res[res["model"] == "Persistence"].set_index("h")["rmse"]
    res["skill"] = 1 - res["rmse"] / res["h"].map(base)
    res.insert(0, "target", target)
    info = {"target": target, "n_train": len(train), "n_test": len(test),
            "cutoff": cutoff, "test_ponds": test["series"].nunique()}
    return res, info, test, preds, y, pd.DataFrame(alerts)


def useful_lead_time(res: pd.DataFrame, cfg: PrepConfig, min_r2: float):
    """Best model = lowest mean RMSE over all horizons. Its lead time is the
    longest h for which every step up to h has R² >= min_r2 and skill > 0."""
    trained = res[res["model"] != "Persistence"]
    best = trained.groupby("model")["rmse"].mean().idxmin()
    b = trained[trained["model"] == best].sort_values("h")
    ok = (b["r2"] >= min_r2) & (b["skill"] > 0)
    steps = int(ok.cumprod().sum())
    return best, steps * cfg.freq_min / 60


def plot_rmse(all_res: pd.DataFrame, cfg: PrepConfig, out_dir: str) -> str:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    targets = all_res["target"].unique()
    fig, axes = plt.subplots(1, len(targets), figsize=(4.2 * len(targets), 3.6))
    for ax, t in zip(np.atleast_1d(axes), targets):
        for model, g in all_res[all_res["target"] == t].groupby("model", sort=False):
            ax.plot(g["h"] * cfg.freq_min / 60, g["rmse"], marker=".", label=model)
        ax.set_title(PRETTY[t])
        ax.set_xlabel("hours ahead")
        ax.grid(alpha=0.3)
    np.atleast_1d(axes)[0].set_ylabel("RMSE (test)")
    np.atleast_1d(axes)[0].legend(fontsize=8)
    fig.tight_layout()
    path = os.path.join(out_dir, "rmse_vs_horizon.png")
    fig.savefig(path, dpi=120)
    plt.close(fig)
    return path


def plot_example(target, test, preds, y, cfg, out_dir, h_hours=(1, 2, 3)) -> str:
    """Actual vs forecast made h hours earlier, for the test pond with most rows."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    pond = test["series"].value_counts().idxmax()
    sel = (test["series"] == pond).to_numpy()
    t = test.loc[sel, "time"]
    # one panel per distinct step that fits in the horizon (10-min horizon -> 1 panel)
    steps = sorted({min(max(int(hh * 60 / cfg.freq_min), 1), cfg.horizon_steps) for hh in h_hours})
    fig, axes = plt.subplots(len(steps), 1, figsize=(12, 2.6 * len(steps)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, h in zip(axes, steps):
        when = (t + pd.Timedelta(minutes=h * cfg.freq_min)).to_numpy()
        order = np.argsort(when)[:6 * 24 * 60 // cfg.freq_min // 6 * 3]      # ~3 days of rows
        ax.plot(when[order], y[sel, h - 1][order], ".", ms=2, color="#222", label="actual")
        for name, p in preds.items():
            if name != "persistence":
                ax.plot(when[order], p[sel, h - 1][order], ".", ms=2, label=name)
        ax.set_ylabel(f"+{h * cfg.freq_min} min" if h * cfg.freq_min < 60
                      else f"+{h * cfg.freq_min / 60:g} h")
        ax.grid(alpha=0.3)
    axes[0].set_title(f"{PRETTY[target]} — forecast vs actual, test set, {pond}")
    axes[0].legend(fontsize=8, markerscale=4)
    fig.tight_layout()
    path = os.path.join(out_dir, f"forecast_example_{target}.png")
    fig.savefig(path, dpi=120)
    plt.close(fig)
    return path


def main() -> None:
    ap = argparse.ArgumentParser(description="AquaSense forecasting: model comparison")
    ap.add_argument("--data", default=DATA_DIR)
    ap.add_argument("--db")
    ap.add_argument("--tank")
    ap.add_argument("--freq", type=int, default=PrepConfig.freq_min)
    ap.add_argument("--lags", type=int, default=PrepConfig.lag_steps)
    ap.add_argument("--horizon", type=int, default=PrepConfig.horizon_steps)
    ap.add_argument("--models", default=",".join(MODELS))
    ap.add_argument("--targets", default=",".join(PrepConfig().params))
    ap.add_argument("--min-r2", type=float, default=0.7,
                    help="R² a horizon step needs to count as reliable")
    ap.add_argument("--out", default=OUT_DIR)
    args = ap.parse_args()
    cfg = PrepConfig(freq_min=args.freq, lag_steps=args.lags, horizon_steps=args.horizon)
    model_names = args.models.split(",")
    if "persistence" not in model_names:
        model_names.insert(0, "persistence")       # needed for the skill score
    os.makedirs(args.out, exist_ok=True)
    pd.set_option("display.width", 200)

    raw = load_controller_db(args.db, args.tank) if args.db else load_csv_dir(args.data)
    grid, _ = prepare(raw, cfg)

    limits = load_safe_ranges()
    all_res, infos, lead, all_alerts = [], [], [], []
    for target in args.targets.split(","):
        res, info, test, preds, y, al = evaluate_target(grid, target, cfg, model_names,
                                                        limits[target])
        all_res.append(res)
        all_alerts.append(al)
        infos.append(info)
        best, hours = useful_lead_time(res, cfg, args.min_r2)
        lead.append({"target": target, "best_model": best, "useful_lead_h": hours})
        plot_example(target, test, preds, y, cfg, args.out)
        print(f"[{PRETTY[target]}] train={info['n_train']:,} test={info['n_test']:,} "
              f"({info['test_ponds']} ponds) cutoff={info['cutoff']:%Y-%m-%d}")
    all_res = pd.concat(all_res, ignore_index=True)
    all_res.to_csv(os.path.join(args.out, "metrics_by_horizon.csv"), index=False)

    show = sorted({1, 60 // cfg.freq_min, 120 // cfg.freq_min, cfg.horizon_steps}
                  & set(range(1, cfg.horizon_steps + 1)))
    for metric in ["rmse", "mae", "mape_%", "r2", "skill"]:
        tbl = all_res[all_res["h"].isin(show)].pivot_table(
            index=["target", "model"], columns="h", values=metric, sort=False)
        tbl.columns = [f"+{h * cfg.freq_min / 60:g}h" for h in tbl.columns]
        print(f"\n=== {metric.upper()} on held-out test data")
        print(tbl.round(3).to_string())

    fit = all_res.groupby(["target", "model"], sort=False)["fit_s"].first().unstack()
    print("\n=== fit time (s)\n" + fit.to_string())
    lead = pd.DataFrame(lead)
    print(f"\n=== useful lead time (best model; every step up to it has R² >= {args.min_r2} "
          f"and beats persistence)\n" + lead.to_string(index=False))
    lead.to_csv(os.path.join(args.out, "lead_time.csv"), index=False)

    all_alerts = pd.concat(all_alerts, ignore_index=True)
    all_alerts.to_csv(os.path.join(args.out, "alert_metrics.csv"), index=False)
    print(f"\n=== early alerts on held-out data (limits from controller/config.yaml, "
          f"breach = {CONFIRM_STEPS} consecutive steps, horizon "
          f"{cfg.horizon_steps * cfg.freq_min / 60:g} h)")
    print(all_alerts.round(2).to_string(index=False))
    print(f"\nwrote {plot_rmse(all_res, cfg, args.out)} and metrics_by_horizon.csv")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile prediction/train.py
"""Train the deployed forecasters on ALL available data and save them.

evaluate.py decides which model to trust (held-out test); this script refits
that choice on everything, so the live service uses every row we have.

Run:
    python -m prediction.train                                  # IoTPond CSVs
    python -m prediction.train --db controller/controller.db    # your own logs
"""
from __future__ import annotations

import argparse
import datetime as dt
import os

import joblib

from .alerts import load_safe_ranges, tune_margin
from .data_prep import (DATA_DIR, OUT_DIR, label_matrix, load_controller_db, load_csv_dir,
                        make_lag_dataset, prepare)
from .evaluate import time_split
from .models import MODELS
from .settings import PRETTY, PrepConfig

MODEL_PATH = os.path.join(OUT_DIR, "models", "forecaster.joblib")

# Chosen from the held-out comparison (see evaluate.py / README): gradient
# boosting gave the best mix of alert precision and warned breaches for every
# parameter. Change one name here to deploy another model for a parameter.
DEPLOYED = {"temperature": "gboost", "ammonia": "gboost", "do": "gboost", "ph": "gboost"}


def train_all(grid, cfg: PrepConfig, choice: dict[str, str],
              limits: dict | None = None) -> tuple[dict, dict]:
    """Returns (models, alert margins). Each margin is tuned for F1 on the
    latest 20% of the data with a model fitted on the rest; the deployed model
    is then refitted on everything."""
    limits = limits or load_safe_ranges()
    models, margins = {}, {}
    for target, name in choice.items():
        sup = make_lag_dataset(grid, target, cfg)
        inner, val, _ = time_split(sup, cfg)
        tuner = MODELS[name](cfg).fit(inner, target)
        margins[target], f1 = tune_margin(val, label_matrix(val, cfg), tuner.predict(val),
                                          target, limits[target], cfg)
        models[target] = MODELS[name](cfg).fit(sup, target)
        print(f"  {PRETTY[target]:<17} {models[target].name:<12} trained on {len(sup):,} rows  "
              f"alert margin {margins[target]:+.3f} (validation F1 {f1:.2f})")
    return models, margins


def main() -> None:
    ap = argparse.ArgumentParser(description="AquaSense forecasting: train + save")
    ap.add_argument("--data", default=DATA_DIR)
    ap.add_argument("--db")
    ap.add_argument("--tank")
    ap.add_argument("--freq", type=int, default=PrepConfig.freq_min)
    ap.add_argument("--lags", type=int, default=PrepConfig.lag_steps)
    ap.add_argument("--horizon", type=int, default=PrepConfig.horizon_steps)
    ap.add_argument("--out", default=MODEL_PATH)
    args = ap.parse_args()
    cfg = PrepConfig(freq_min=args.freq, lag_steps=args.lags, horizon_steps=args.horizon)

    raw = load_controller_db(args.db, args.tank) if args.db else load_csv_dir(args.data)
    grid, _ = prepare(raw, cfg)
    models, margins = train_all(grid, cfg, DEPLOYED)

    os.makedirs(os.path.dirname(args.out), exist_ok=True)
    joblib.dump({"cfg": cfg, "models": models, "margins": margins,
                 "trained_at": dt.datetime.now().isoformat(timespec="seconds"),
                 "trained_on": args.db or args.data}, args.out, compress=3)
    print(f"wrote {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile controller/config.yaml
# =============================================================================
# AquaSense — Threshold-based reactive BASELINE controller configuration
# =============================================================================
# Everything that differs between "today" (temperature only, no actuators)
# and "later" (all sensors + real hardware) lives in this file:
#
#   parameters.<name>.source   payload   -> value must come from the device JSON
#                              simulated -> always generated by the server
#                              auto      -> device value if present, else simulated
#                              derived   -> computed from other parameters
#
#   actuators.<name>.driver    mock       -> log ON/OFF only (no hardware)
#                              esp32_http -> send command to the ESP32 over HTTP
#
# Swapping a mock to a real device = change one word here. No code changes.
# =============================================================================

database: controller.db        # SQLite file (relative to this config file).
                               # Use a fresh DB per experiment run, e.g.
                               #   AQUASENSE_DB=runs/baseline_pond1.db

default_tank: tank1
default_mode: AUTO             # AUTO = engine drives actuators, MANUAL = user does
engine: threshold              # the control strategy (baseline). The predictive
                               # controller will be a second engine name here.

data_gap_s: 900                # watchdog: if a tank sends nothing for this long,
                               # running actuators are logged OFF and alerts are
                               # closed at (last reading + data_gap_s). Stops data
                               # outages (the IoTPond CSVs have days-long gaps)
                               # from inflating run-time. null = disabled.

mqtt:
  enabled: false               # set true to also ingest via MQTT
  host: localhost
  port: 1883
  topic: aquasense/+/readings  # '+' = tank_id; payload is the same JSON as REST

# -----------------------------------------------------------------------------
# Water-quality parameters
#   min / max        SAFE RANGE (null = no limit on that side)
#   deadband         hysteresis for actuator switch-OFF (0 = pure on/off)
#   critical_margin  distance beyond min/max at which WARNING becomes CRITICAL
#   valid_min/max    physically plausible range; values outside are treated as
#                    SENSOR FAULTS (e.g. DS18B20 returns -127 when unplugged)
#                    and are never used for control.
#   simulation       crude plant model used when source is simulated/auto
#     start, mean        initial value and the level the water drifts back to
#     reversion          fraction of the gap to `mean` closed per minute
#     drift_per_min      natural trend (e.g. fish respiration consumes DO)
#     noise              std-dev of measurement noise
#     effects            how actuators move the value
#                          per_minute: while the actuator is ON
#                          per_pulse:  each time the actuator is pulsed
# -----------------------------------------------------------------------------
parameters:
  temperature:
    label: Temperature
    unit: "°C"
    source: payload            # ESP32 + DS18B20 is connected for this one
    min: 24.0
    max: 28.0
    deadband: 0.3
    critical_margin: 2.0
    valid_min: -5.0
    valid_max: 50.0
    simulation:
      start: 25.5
      mean: 25.5
      reversion: 0.01
      drift_per_min: -0.01
      noise: 0.03
      effects:
        heater: { per_minute: 0.05 }

  do:
    label: Dissolved oxygen
    unit: mg/L
    source: auto               # not connected yet -> simulated unless sent
    min: 5.0
    max: null
    deadband: 0.3
    critical_margin: 1.5
    valid_min: 0.0
    valid_max: 25.0
    simulation:
      start: 6.5
      mean: 6.5
      reversion: 0.01
      drift_per_min: -0.03     # respiration pulls DO below 5 -> aerator demo
      noise: 0.05
      effects:
        aerator: { per_minute: 0.15 }

  ph:
    label: pH
    unit: pH
    source: auto
    min: 6.5
    max: 8.0
    deadband: 0.1
    critical_margin: 0.5
    valid_min: 0.0
    valid_max: 14.0
    simulation:
      start: 7.0
      mean: 7.0
      reversion: 0.002
      drift_per_min: -0.003    # nitrification slowly acidifies the water
      noise: 0.01
      effects:
        dosing_pump: { per_pulse: 0.08 }   # buffer dose raises pH

  ammonia:
    label: Ammonia (TAN)
    unit: mg/L
    source: auto
    min: null
    max: 0.5                   # total ammonia nitrogen; toxicity is driven by
    deadband: 0.05             # the unionised fraction (see nh3 below)
    critical_margin: 0.5
    valid_min: 0.0
    valid_max: 50.0
    simulation:
      start: 0.15
      mean: 0.15
      reversion: 0.005
      drift_per_min: 0.0004
      noise: 0.005

  nh3:
    label: Unionised NH₃
    unit: mg/L
    source: derived            # computed from ammonia + pH + temperature
    derive: unionised_ammonia
    min: null
    max: 0.05
    deadband: 0.005
    critical_margin: 0.05
    valid_min: 0.0
    valid_max: 50.0

# -----------------------------------------------------------------------------
# Actuators (instantiated once per tank)
#   power_w  rated power, used to turn run-time into an energy estimate (Wh)
#   url      only for driver esp32_http: base URL of the ESP32 command server
# -----------------------------------------------------------------------------
actuators:
  aerator:
    label: Aerator
    driver: mock
    power_w: 35
  heater:
    label: Heater
    driver: mock
    power_w: 100
  dosing_pump:
    label: pH dosing pump
    driver: mock
    power_w: 6
    # driver: esp32_http
    # url: http://192.168.1.50

# -----------------------------------------------------------------------------
# Reactive control rules (the BASELINE logic)
#   condition  below_min | above_max | out_of_range
#   action     hold  -> ON while condition holds, OFF once the value is back
#                       inside the range by `deadband` (hysteresis)
#              pulse -> run for pulse_seconds, then lock out for cooldown_seconds
# Parameters with no rule (e.g. high temperature, ammonia) only raise alerts.
# -----------------------------------------------------------------------------
rules:
  - parameter: do
    condition: below_min
    actuator: aerator
    action: hold

  - parameter: temperature
    condition: below_min
    actuator: heater
    action: hold

  - parameter: ph
    condition: out_of_range
    actuator: dosing_pump
    action: pulse
    pulse_seconds: 5
    cooldown_seconds: 600

## 2. Settings

Change these and re-run from here. One step = `FREQ_MIN` minutes:
`HORIZON_STEPS = 1` → 10 min, `6` → 1 h, `18` → 3 h.

In [ ]:
import importlib, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

import prediction.settings, prediction.data_prep, prediction.models, prediction.alerts, prediction.evaluate
for m in [prediction.settings, prediction.data_prep, prediction.models, prediction.alerts, prediction.evaluate]:
    importlib.reload(m)   # pick up edits if the code cells were re-run
from prediction.settings import PrepConfig, PRETTY
from prediction.data_prep import load_csv_dir, prepare, grid_report, make_lag_dataset, plot_before_after, OUT_DIR
from prediction.evaluate import evaluate_target, useful_lead_time, plot_rmse, plot_example
from prediction.alerts import load_safe_ranges

FREQ_MIN      = 10    # resample interval (minutes)
LAG_STEPS     = 12    # past readings used as input (12 x 10 min = 2 h)
HORIZON_STEPS = 1     # steps forecast ahead (1 x 10 min = 10 min)
MIN_R2        = 0.7   # R² a horizon step needs to count as "reliable"
MODEL_NAMES   = ["persistence", "trend", "ridge", "gboost"]

cfg = PrepConfig(freq_min=FREQ_MIN, lag_steps=LAG_STEPS, horizon_steps=HORIZON_STEPS)
limits = load_safe_ranges()
os.makedirs(OUT_DIR, exist_ok=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print(cfg)
print("safe ranges:", {p: limits[p] for p in cfg.params})

## 3. Step 1 — Data preparation

Implausible values and stuck sensors → missing; copied rows between ponds dropped;
10-min median bins; spike filter; gaps ≤ 30 min filled; light trailing smoothing.

In [ ]:
raw = load_csv_dir("data")
print(f"{len(raw):,} raw rows from {raw['series'].nunique()} ponds")
grid, report = prepare(raw, cfg)

print("\nRemoved per pond (% of that pond's raw rows):")
display(report)
print("Clean 10-min grid — coverage % per parameter:")
display(grid_report(grid, cfg).round(1))

rows = {PRETTY[p]: len(make_lag_dataset(grid, p, cfg)) for p in cfg.params}
print("Usable training rows per parameter:", rows)
display(grid.dropna(subset=cfg.params).head(8))

In [ ]:
best = grid.dropna(subset=cfg.params).groupby("series").size().idxmax()
display(Image(plot_before_after(raw, grid, cfg, best)))

## 4. Steps 2 + 4 — Train on the older 80 %, test on the latest 20 %

One date cutoff for all ponds; training labels never reach into the test period.

In [ ]:
all_res, all_alerts, lead, tests = [], [], [], {}
for target in cfg.params:
    t0 = time.time()
    res, info, test, preds, y, al = evaluate_target(grid, target, cfg, MODEL_NAMES, limits[target])
    all_res.append(res); all_alerts.append(al); tests[target] = (test, preds, y)
    best_model, hours = useful_lead_time(res, cfg, MIN_R2)
    lead.append({"parameter": PRETTY[target], "best_model": best_model, "reliable_up_to_h": round(hours, 2)})
    print(f"{PRETTY[target]:<17} train={info['n_train']:>6,}  test={info['n_test']:>6,}  "
          f"cutoff={info['cutoff']:%Y-%m-%d}  ({time.time() - t0:.0f} s)")
all_res = pd.concat(all_res, ignore_index=True)
all_alerts = pd.concat(all_alerts, ignore_index=True)
all_res.to_csv(os.path.join(OUT_DIR, "metrics_by_horizon.csv"), index=False)
all_alerts.to_csv(os.path.join(OUT_DIR, "alert_metrics.csv"), index=False)

### Forecast accuracy per parameter and model

**Skill** = 1 − RMSE / RMSE(persistence): above 0 means better than assuming "nothing changes".

In [ ]:
show = sorted({1, 60 // cfg.freq_min, 120 // cfg.freq_min, cfg.horizon_steps} & set(range(1, cfg.horizon_steps + 1)))
label = lambda h: f"+{h * cfg.freq_min} min" if h * cfg.freq_min < 60 else f"+{h * cfg.freq_min / 60:g} h"
for metric in ["rmse", "mae", "mape_%", "r2", "skill"]:
    tbl = all_res[all_res["h"].isin(show)].pivot_table(index=["target", "model"], columns="h",
                                                        values=metric, sort=False)
    tbl.columns = [label(h) for h in tbl.columns]
    print(f"\n=== {metric.upper()}")
    display(tbl.round(3))

print("Useful lead time (every step up to it has R² >= MIN_R2 and beats persistence):")
display(pd.DataFrame(lead))

## 5. Early-alert scores (with F1)

Counted only while the water is still safe (otherwise there is nothing to warn about early).

* **precision**: share of alerts followed by a real breach
* **recall**: share of real coming breaches that got an alert
* **f1** = 2·P·R / (P + R): high only when both are high
* **warned_%**: share of distinct breaches warned at least once; **lead_med_h**: median earliest warning
* **(perfect forecast)**: the best any model could do on this data
* **margin**: the alert line is moved this far inside the safe limit (negative = outside),
  chosen for the best F1 on a validation slice (latest 20 % of training data, never the test set).
  **f1_untuned** is the same model alerting exactly at the limit.

In [ ]:
cols = ["target", "model", "margin", "alerts", "precision", "recall", "f1", "f1_untuned",
        "events", "warned_%", "lead_med_h", "warned_1h_%"]
display(all_alerts[cols].round(2))

f1 = all_alerts[~all_alerts["model"].isin(["Persistence", "(perfect forecast)"])].pivot(
    index="target", columns="model", values="f1")
ax = f1.rename(index=PRETTY).plot.bar(figsize=(9, 3.6), rot=0, width=0.75)
ax.set_ylabel("F1 (early alerts)"); ax.set_ylim(0, 1); ax.set_xlabel("")
ax.set_title(f"Early-alert F1 per model — horizon {label(cfg.horizon_steps)}")
ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

## 6. Plots — forecast vs actual on the test set

In [ ]:
if cfg.horizon_steps > 1:
    display(Image(plot_rmse(all_res, cfg, OUT_DIR)))
for target, (test, preds, y) in tests.items():
    display(Image(plot_example(target, test, preds, y, cfg, OUT_DIR)))

## 7. Train the final models on all data and download

Put the downloaded files into the project:

* `forecaster.joblib` → `water_quality/outputs/prediction/models/forecaster.joblib`
* `alert_metrics.csv` → `water_quality/outputs/prediction/alert_metrics.csv` (the dashboard's reliability notes)

Then restart the forecasting service (`python -m prediction.service`).

In [ ]:
import datetime as dt, joblib
from prediction.train import DEPLOYED, train_all

DEPLOYED = dict(DEPLOYED)   # e.g. DEPLOYED["do"] = "trend" to deploy another model for DO
models, margins = train_all(grid, cfg, DEPLOYED, limits)
path = os.path.join(OUT_DIR, "forecaster.joblib")
joblib.dump({"cfg": cfg, "models": models, "margins": margins,
             "trained_at": dt.datetime.now().isoformat(timespec="seconds"),
             "trained_on": "IoTPond CSVs (Colab)"}, path, compress=3)
print("saved", path, f"({os.path.getsize(path) / 1e6:.1f} MB)")

from google.colab import files
files.download(path)
files.download(os.path.join(OUT_DIR, "alert_metrics.csv"))